# Lesson 01 - AI 代理简介

欢迎来到<strong>AI代理初学者</strong>课程的第一课！

<strong>AI代理</strong>是一个使用大型语言模型（LLM）作为其推理引擎的程序，可以在现实世界中采取<em>动作</em>——调用API、查询数据库或运行代码——以代表用户完成某个目标。

在本笔记本中，您将构建您的第一个代理：一个推荐度假目的地的<strong>旅行代理</strong>。在此过程中，您将学习如何：

1. 使用<strong>Microsoft代理框架</strong>连接到Azure AI Foundry代理服务。
2. 给代理添加一个<strong>工具</strong>——它可以调用的普通Python函数。
3. 运行代理并检查其响应。
4. 逐令牌流式传输代理的响应。


## 设置

在运行此笔记本之前，请确保您已完成以下操作：

1. **拥有一个部署了聊天模型（例如 `gpt-4o-mini`）的 Azure AI Foundry 项目**。
2. **通过 Azure CLI 登录** — 在终端运行 `az login`。
3. **设置必需的环境变量：**
   - `AZURE_AI_PROJECT_ENDPOINT` — 您的 Azure AI Foundry 项目终结点。
   - `AZURE_AI_MODEL_DEPLOYMENT_NAME` — 您部署的模型名称。

下面的单元格将安装您需要的 Python 包。


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import dotenv
from agent_framework.foundry import FoundryChatClient
from azure.identity import AzureCliCredential
from agent_framework import tool

dotenv.load_dotenv(dotenv.find_dotenv())

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
model = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

if not endpoint or not model:
    raise ValueError(
        "Missing required environment variables. "
        "Please set AZURE_AI_PROJECT_ENDPOINT and AZURE_AI_MODEL_DEPLOYMENT_NAME in your .env file."
    )

provider = FoundryChatClient(
    project_endpoint=endpoint,
    model=model,
    credential=AzureCliCredential()
)

## 创建您的第一个代理

代理需要两样东西：

- <strong>指令</strong>，告诉它<em>它是谁</em>以及<em>如何表现</em>（系统提示）。
- <strong>工具</strong> — 使用 `@tool` 装饰的 Python 函数，代理可以调用这些函数来获取信息或执行操作。

下面我们定义了一个简单的工具，返回一个受欢迎的度假目的地列表。当用户询问旅行推荐时，代理将使用此工具。


In [ ]:
@tool(approval_mode="never_require")
def get_destinations() -> list[str]:
    """Get a list of popular vacation destinations."""
    return [
        "Barcelona",
        "Paris",
        "Berlin",
        "Tokyo",
        "Sydney",
        "New York City",
        "Cairo",
        "Cape Town",
        "Rio de Janeiro",
        "Bali",
    ]

In [ ]:
agent = provider.as_agent(
    name="TravelAgent",
    instructions=(
        "You are a helpful travel agent. Help users find their perfect vacation "
        "destination based on their preferences. Use the get_destinations tool "
        "to see available destinations."
    ),
    tools=[get_destinations],
)

response = await agent.run(
    "I'm looking for a warm beach destination. What do you recommend?"
)
print(response)

## 流式响应

为了获得更互动的体验，您可以<strong>流式传输</strong>代理的响应。代理会在生成文本块时逐步输出，而不是等待完整回复。这在聊天界面中特别有用，因为您可以实时显示输出。


In [ ]:
async for chunk in agent.run(
    "Tell me about Tokyo as a travel destination", stream=True
):
    print(chunk, end="", flush=True)

## Summary

在本课中，您学习了如何：

- <strong>创建一个提供者</strong>，通过 `AzureAIProjectAgentProvider` 连接到 Azure AI Foundry Agent 服务。
- **使用 `@tool` 装饰器定义工具**，以便代理可以调用您的 Python 函数。
- <strong>运行代理</strong>，处理用户消息并打印其响应。
- <strong>流式传输响应</strong>，实现实时输出。

在下一课中，我们将更深入地探索代理框架，学习如何赋予代理更强大的工具和多步推理能力。


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**免责声明**：
本文件由 AI 翻译服务 [Co-op Translator](https://github.com/Azure/co-op-translator) 翻译完成。尽管我们力求准确，但请注意，自动翻译可能包含错误或不准确之处。原始语言版文件应视为权威来源。对于重要信息，建议使用专业人工翻译。我们对因使用本翻译而产生的任何误解或误释不承担责任。
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
